In [2]:
import torch
import glob
import os
from tqdm import tqdm
from transformers import pipeline
import json

# --- 1. เตรียม "ไม้กายสิทธิ์" (Load the Model using Pipeline) ---
# ตรวจสอบว่ามี GPU (NVIDIA CUDA) ให้ใช้ไหม ถ้ามีจะเร็วขึ้นมาก!
device = "cuda:1" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading model 'openai/whisper-large-v3' onto device: {device}")

# สร้าง pipeline เหมือนเรียกใช้เวทมนตร์อัตโนมัติ
# มันจะจัดการเรื่อง processor และ model ให้เราเองเลยค่ะ
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch_dtype,
    device=device,
)

print("Model loaded successfully!")


# --- 2. หา "วัตถุดิบ" ทั้งหมด (Find All Audio Files) ---
# Path ไปยังโฟลเดอร์ที่เก็บไฟล์เสียงของเรา (ใช้ ../ เพื่อถอยกลับไปหนึ่งขั้น)
audio_folder_path = "../final_audio_output_final/"
wav_files = glob.glob(os.path.join(audio_folder_path, "*.wav"))
print(f"Found {len(wav_files)} audio files to process in '{audio_folder_path}'")


# --- 3. เริ่ม "ร่ายเวทมนตร์" ทีละไฟล์ (Process Each File) ---
results = []
# ใช้ tqdm เพื่อให้เห็น progress bar สวยๆ ตอนทำงาน
for audio_path in tqdm(wav_files, desc="Transcribing audio files"):
    try:
        # ส่ง path ของไฟล์เสียงให้ pipeline จัดการได้เลย!
        transcription_result = pipe(audio_path)
        
        # ผลลัพธ์ที่ได้จะเป็น Dictionary หน้าตาประมาณ {'text': 'สวัสดีครับ...'}
        transcript_text = transcription_result["text"]

        # เก็บชื่อไฟล์และข้อความที่ถอดเสียงได้
        file_name = os.path.basename(audio_path)
        results.append({"file": file_name, "transcript": transcript_text})

    except Exception as e:
        file_name = os.path.basename(audio_path)
        results.append({"file": file_name, "transcript": f"ERROR: {e}"})


# --- 4. แสดงผลลัพธ์ทั้งหมด (ส่วนอัปเกรด) ---
print("\n--- Transcription Complete! ---\n")
for result in results:
    # เพิ่ม .strip() เพื่อตัดช่องว่างที่ไม่จำเป็นหน้า-หลังข้อความออกค่ะ
    print(f"File: {result['file']} -> Transcript: {result['transcript'].strip()}\n")

# --- 5. บันทึกผลลัพธ์ลงม้วนคัมภีร์! (ส่วนที่เพิ่มใหม่) ---
output_filename = "transcription_results.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"✅ Results successfully saved to '{output_filename}'")

Loading model 'openai/whisper-large-v3' onto device: cuda:1


Device set to use cuda:1


Model loaded successfully!
Found 300 audio files to process in '../final_audio_output_final/'


Transcribing audio files:   0%|          | 0/300 [00:00<?, ?it/s]`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
Transcribing audio files: 100%|██████████| 300/300 [03:10<00:00,  1.58it/s]


--- Transcription Complete! ---

File: dialogue_16_utterance_1.wav -> Transcript: I'm feeling pretty anxious, to be honest. I've been really worried about not being able to see my brothers.

File: dialogue_9_utterance_9.wav -> Transcript: Not really. It feels like it would be too risky to put myself out there just to get hurt again.

File: dialogue_14_utterance_7.wav -> Transcript: I start imagining they'll forget about me. World drift apart so much that we won't be close anymore. It makes me feel really helpless and scared.

File: dialogue_3_utterance_11.wav -> Transcript: I guess if I could manage my anxiety better and remind myself that the animals aren't mad at me, maybe if I can shift my focus and not be so hard on myself.

File: dialogue_12_utterance_16.wav -> Transcript: But I don't know if it's realistic.

File: dialogue_12_utterance_11.wav -> Transcript: Not really. Most people I talk to about it just don't get it. They think it's silly or strange.

File: dialogue_16_utteranc